In [10]:
import requests
import torch

from torch.utils.data import TensorDataset, DataLoader

In [11]:
# Loading the dataset

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

vocab = sorted(set(text))
vocab_size = len(vocab)

# Keep tensor creation on CPU (DataLoader shuffle breaks if default is MPS).
torch.set_default_device("cpu")
torch.set_float32_matmul_precision("high")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [12]:
# Tokenizer

stoi = {ch: i for i, ch in enumerate(vocab)}  # Char to Int
itos = {i: ch for i, ch in enumerate(vocab)}  # Int to Char

encode = lambda s: [stoi[c] for c in s]  # String -> List[int]
decode = lambda v: "".join([itos[i] for i in v])  # List[Int] -> String

# Encoding the data
encoded_text = torch.tensor(encode(text), dtype=torch.long)

In [34]:
batch_size = 256
seq_len = 128

N = (len(encoded_text) - 1) // seq_len

chunks = encoded_text[:N * seq_len + 1]

X = chunks[:-1].reshape(N, seq_len)
y = chunks[1:].reshape(N, seq_len)

D = int(0.9 * len(X))

X_train, y_train = X[:D], y[:D]
X_test, y_test = X[D:], y[D:]

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [35]:
# Positional encodings
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, seq_len, embed_dim):
        super().__init__()
        PE = torch.zeros(seq_len, embed_dim)

        pos = torch.arange(seq_len).unsqueeze(1)        # (seq_len, 1)
        div = torch.exp(
            torch.arange(0, embed_dim, 2) * (-math.log(10000) / embed_dim)
        )                                                    # (embed_dim/2,)

        PE[:, 0::2] = torch.sin(pos * div)                  # even dims
        PE[:, 1::2] = torch.cos(pos * div)                  # odd dims

        self.register_buffer("PE", PE.unsqueeze(0))        

    def forward(self, x):
        return x + self.PE[:, :x.shape[1], :]

In [36]:


embed_dim = 128
num_heads = 4
num_layers = 4
ff_dim = 512


class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Positional Encoding
        self.positional_encoding = PositionalEncoding(
            seq_len=seq_len, embed_dim=embed_dim
        )
        # Transformer layer
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            norm_first=True,
            batch_first=True,
        )
        # Transformer
        self.transformer = nn.TransformerEncoder(
            encoder_layer=layer, num_layers=num_layers, norm=nn.LayerNorm(embed_dim)
        )

    def forward(self, X):
        x = self.embedding(X)  
        x = self.positional_encoding.forward(x)
        T = X.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=X.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        x = x @ self.embedding.weight.T
        return x

In [37]:
import torch.optim as optim
from torch.nn import functional as F

model = Transformer().to(device)

/var/folders/fy/tqc62nzj71n45pm6tkjtbppm0000gn/T/ipykernel_91484/1149722155.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [41]:


optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        logits = model(X)

        logits = logits.view(-1, vocab_size)
        targets = y.view(-1)
        loss = F.cross_entropy(logits.float(), targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_loader):.4f}")

Epoch 1: loss = 2.0142
Epoch 2: loss = 1.8897
Epoch 3: loss = 1.8749
Epoch 4: loss = 1.8628
Epoch 5: loss = 1.8502
Epoch 6: loss = 1.8389
Epoch 7: loss = 1.8270
Epoch 8: loss = 1.8156
Epoch 9: loss = 1.8042
Epoch 10: loss = 1.7934
Epoch 11: loss = 1.7801
Epoch 12: loss = 1.7696
Epoch 13: loss = 1.7578
Epoch 14: loss = 1.7490
Epoch 15: loss = 1.7384
Epoch 16: loss = 1.7289
Epoch 17: loss = 1.7196
Epoch 18: loss = 1.7104
Epoch 19: loss = 1.7030
Epoch 20: loss = 1.6934


In [18]:
# Generate
def gen(prompt, num_tokens):
    print("Original text\n", prompt)
    tokens = torch.tensor(encode(prompt), dtype=torch.long)

    model.eval()
    with torch.no_grad():
        for _ in range(num_tokens):
            X = tokens[-seq_len:].unsqueeze(0).to(device)
            prob = F.softmax(model(X), dim=-1)[0, -1].cpu()
            sample = torch.multinomial(prob, num_samples=1)
            tokens = torch.cat((tokens, sample))

    print("Generated text is:\n", decode(tokens.tolist()))

In [44]:
gen(text[:200], 500)

Original text
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
Generated text is:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you no shoild;
More brant the earthsolard: leave it the vaers.

Senvant:
Sho 'tis bur I canon mean: know I long and at for
have and knaws, injetion Engeorth he your sealty?
O no it visle: whee'll made anst tone.

CORIOLANUS:
At we honour you not your alive leave batterch:
What his the doot by how not? I will not,
When saigh yet God, look's night nKean,
I hand serve subject hotss you; I will be it of this,
And foor old make those ruys from that the shere?

First he druck:
If what his with in despoce
